# Document Extraction and ChromaDB Embeddings

This notebook runs the repository's document extraction and embedding flow in Google Colab.
It stops after embeddings are stored and verified in ChromaDB. It does not run the application-analysis agents, semantic search, or RAG.

For restricted data, use synthetic documents only. Ollama is used locally in this Colab runtime for image and scanned-document OCR.

## 1. Clone the repository

In [ ]:
REPO_URL = "https://github.com/Govindkm/tcs-ai-club-hackathon-prompt-pioneers.git"
BRANCH = "develop"
PROJECT_DIR = "/content/tcs-ai-club-hackathon-prompt-pioneers"

import os

if not os.path.isdir(PROJECT_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    print("Repository already exists.")

%cd {PROJECT_DIR}

## 2. Install required Python packages

This installs the repository dependencies, including `chromadb` and `sentence-transformers`.

In [ ]:
!pip install -q -r requirements.txt

## 3. Install Ollama

Ollama is required only when extraction needs OCR for images, scanned PDF pages, or embedded document images.

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

## 4. Start Ollama and pull the OCR model

In [ ]:
import os
import subprocess
import time
import requests

OLLAMA_HOST = "http://127.0.0.1:11434"
TEXT_MODEL_NAME = "llama3.2:3b"
VISION_MODEL_NAME = "minicpm-v"
os.environ["OLLAMA_HOST"] = OLLAMA_HOST
os.environ["OLLAMA_MODEL_ID"] = TEXT_MODEL_NAME
os.environ["OLLAMA_VISION_MODEL"] = VISION_MODEL_NAME

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for attempt in range(30):
    try:
        response = requests.get(OLLAMA_HOST, timeout=2)
        if response.status_code == 200:
            print("Ollama server is running.")
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start in time.")

!ollama pull {TEXT_MODEL_NAME}
!ollama pull {VISION_MODEL_NAME}

## 5. Configure local ChromaDB and embeddings

The multilingual Sentence Transformers model is downloaded and cached locally on its first use.

In [ ]:
import os

os.environ["CHROMA_PERSIST_DIRECTORY"] = "/content/chroma_db"
os.environ["CHROMA_COLLECTION_NAME"] = "scheme_documents"
os.environ["EMBEDDING_MODEL_NAME"] = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
os.environ["EMBEDDING_CHUNK_SIZE"] = "1000"
os.environ["EMBEDDING_CHUNK_OVERLAP"] = "150"
os.environ["EMBEDDING_BATCH_SIZE"] = "32"

print("Chroma path:", os.environ["CHROMA_PERSIST_DIRECTORY"])
print("Collection:", os.environ["CHROMA_COLLECTION_NAME"])
print("Embedding model:", os.environ["EMBEDDING_MODEL_NAME"])

## 6. Upload multiple documents

In [ ]:
from google.colab import files

uploaded_files = files.upload()
print(f"Uploaded files: {len(uploaded_files)}")

for filename, content in uploaded_files.items():
    print(f"{filename}: {len(content):,} bytes")

## 7. Extract text into structured document records

Each uploaded file becomes a logical document. ZIP entries retain their archive path.

In [ ]:
from src.ingestion.document_reader import extract_documents

file_payloads = list(uploaded_files.items())
documents = extract_documents(file_payloads)

print(f"Extracted logical documents: {len(documents)}")
for index, document in enumerate(documents, start=1):
    print(f"\nDocument {index}")
    print("Title:", document["title"])
    print("Extension:", document["extension"])
    print("Source path:", document["source_path"])
    print("Status:", document["status"])
    print("Content length:", len(document["content"]))
    print("Preview:", document["content"][:250])

## 8. Add optional pasted text as a document

Leave this cell unchanged if you are testing uploaded files only.

In [ ]:
PASTED_TEXT = ""
# Example:
# PASTED_TEXT = "Requested amount: Rs. 250,000. Project: solar micro-grid."

if PASTED_TEXT.strip():
    documents.append(
        {
            "title": "pasted-text.txt",
            "extension": ".txt",
            "source_path": "pasted-text.txt",
            "metadata": '{"source": "colab-pasted-text"}',
            "content": PASTED_TEXT.strip(),
            "status": "extracted",
            "error": None,
        }
    )

print("Documents ready for embedding:", len(documents))

## 9. Inspect the extracted manifest

In [ ]:
import json

print(json.dumps(documents, ensure_ascii=False, indent=2)[:5000])

In [ ]:
import os
import subprocess
import time
import uuid
import requests

# Run the project's real FastAPI backend against a Colab-local SQLite database.
os.environ["APP_DB_PATH"] = "/content/app.db"
os.environ["JWT_SECRET_KEY"] = "colab-only-test-secret"
os.environ["STRANDS_MODEL_PROVIDER"] = "ollama"
os.environ["API_BASE_URL"] = "http://127.0.0.1:8000"

!python scripts/seed_db.py

backend_process = subprocess.Popen(
    ["uvicorn", "backend.app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for attempt in range(30):
    try:
        response = requests.get("http://127.0.0.1:8000/api/v1/health", timeout=2)
        if response.status_code == 200:
            print("FastAPI backend is running.")
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("FastAPI backend did not start in time.")

admin_login = requests.post(
    "http://127.0.0.1:8000/api/v1/auth/login",
    json={"username": "admin", "password": "ChangeMe123!"},
    timeout=30,
)
admin_login.raise_for_status()
admin_headers = {"Authorization": f"Bearer {admin_login.json()['access_token']}"}

schemes_response = requests.get(
    "http://127.0.0.1:8000/api/v1/schemes",
    params={"active_only": "false"},
    headers=admin_headers,
    timeout=30,
)
schemes_response.raise_for_status()
schemes = schemes_response.json()
if not schemes:
    raise RuntimeError("No scheme was found in SQLite.")

scheme = schemes[0]
SCHEME_ID = scheme["id"]
SCHEME_TITLE = scheme["name"]
print("Real SQLite scheme:", SCHEME_ID, SCHEME_TITLE)

## 10. Submit files through the real application API

The previous cell uses the same multipart upload endpoint as the Streamlit application. It creates a real SQLite submission; it does not create a mock submission row.

In [ ]:
print("Real API submission response:")
print(json.dumps(submission, indent=2))
print("SQLite-backed scheme ID:", SCHEME_ID)
print("SQLite-backed scheme title:", SCHEME_TITLE)
print("SQLite-backed submission ID:", SUBMISSION_ID)

## 11. Wait for the real embedding stage

The API background job performs extraction and then invokes the registered `embedding_agent`. This cell waits for that real submission process to finish.

In [ ]:
deadline = time.time() + 600
last_status = None

while time.time() < deadline:
    status_response = requests.get(
        f"http://127.0.0.1:8000/api/v1/applications/{SUBMISSION_ID}/status",
        headers=applicant_headers,
        timeout=30,
    )
    status_response.raise_for_status()
    status = status_response.json()
    current_status = (status["analysis_status"], status.get("analysis_stage"))
    if current_status != last_status:
        print("Analysis status:", current_status)
        last_status = current_status
    if status["analysis_status"] in {"completed", "failed"}:
        break
    time.sleep(5)
else:
    raise TimeoutError("The real application analysis job did not finish in time.")

if status["analysis_status"] == "failed":
    raise RuntimeError(status.get("analysis_error", "The application analysis job failed."))

print("The real submission completed successfully.")

## 12. Verify the number of Chroma records

In [ ]:
import chromadb

verification_client = chromadb.PersistentClient(
    path=os.environ["CHROMA_PERSIST_DIRECTORY"]
)
collection = verification_client.get_collection(
    os.environ["CHROMA_COLLECTION_NAME"]
)

submission_records = collection.get(
    where={"SubmissionID": str(SUBMISSION_ID)},
    include=["documents", "metadatas"],
)
actual_count = len(submission_records["ids"])

print("Chroma records for submission:", actual_count)
assert actual_count > 0
print("Real API submission created and indexed records in ChromaDB.")

## 13. Display stored records and metadata

In [ ]:
records = collection.get(include=["documents", "metadatas"])

for record_id, content, metadata in zip(
    records["ids"], records["documents"], records["metadatas"]
):
    print("=" * 80)
    print("Record ID:", record_id)
    print("Content preview:", content[:300])
    print("Metadata:")
    for key, value in metadata.items():
        print(f"  {key}: {value}")

## 14. Verify scheme and submission metadata

In [ ]:
# Confirm the IDs and title persisted in SQLite match the submitted API record.
from src.db import repository as db

sqlite_submission = db.get_submission(SUBMISSION_ID)
sqlite_scheme = db.get_scheme(sqlite_submission["scheme_id"])

assert sqlite_submission["id"] == SUBMISSION_ID
assert sqlite_submission["scheme_id"] == SCHEME_ID
assert sqlite_scheme["name"] == SCHEME_TITLE
assert len(sqlite_submission["document_manifest"]) > 0

print("SQLite scheme ID:", sqlite_scheme["id"])
print("SQLite scheme title:", sqlite_scheme["name"])
print("SQLite submission ID:", sqlite_submission["id"])
print("SQLite manifest documents:", len(sqlite_submission["document_manifest"]))

## 15. Verify persistence after reopening ChromaDB

In [ ]:
reopened_client = chromadb.PersistentClient(
    path=os.environ["CHROMA_PERSIST_DIRECTORY"]
)
reopened_collection = reopened_client.get_collection(
    os.environ["CHROMA_COLLECTION_NAME"]
)
reopened_records = reopened_collection.get(
    where={"SubmissionID": str(SUBMISSION_ID)},
    include=["documents", "metadatas"],
)

print("Records for submission after reopening:", len(reopened_records["ids"]))
assert len(reopened_records["ids"]) == actual_count
print("Chroma persistence verification passed.")

## 16. Verify the real API submission metadata

In [ ]:
for metadata in reopened_records["metadatas"]:
    assert metadata["SchemeID"] == str(SCHEME_ID)
    assert metadata["SchemeTitle"] == SCHEME_TITLE
    assert metadata["SubmissionID"] == str(SUBMISSION_ID)

print("Chroma metadata matches the real SQLite/API submission.")

## 17. Cleanup

Run this cell when finished. Colab storage is temporary, so the Chroma database will disappear when the runtime is deleted.

In [ ]:
if backend_process.poll() is None:
    backend_process.terminate()
    backend_process.wait(timeout=10)
if ollama_process.poll() is None:
    ollama_process.terminate()
    ollama_process.wait(timeout=10)
print("FastAPI backend and Ollama stopped.")